Notes(What needs to be done): 
1. 
2. 
etc

# 02 - Cleaned Data

**Purpose:** Build the modeling datasets from raw. Reshapes to the two grains
required downstream, fixes types, and validates the result. No analysis, no
plots, no interpretation.

**Input:** `data/raw/CASComp Main DataSet.csv` (unmodified)

**Outputs:**

- `data/processed/frequency_data.csv` - one row per student-coverage (40,000 rows),
  with `claim_count` and `total_amount`. Input to the frequency models.
- `data/processed/claims.csv` - one row per claim (1,819 rows). Input to the
  severity models.

**Depends on:** 00_Inventory, 01_Data_Checks

**This is the only notebook that writes to `data/processed/`.** Everything from
03 onward reads these files and never touches raw. Re-running this notebook
should reproduce both outputs exactly.

**Transformations applied**


**Decisions carried from 01**

- No deductible found, so losses are treated as ground-up with no adjustment
- No censoring at policy limits, so severity is uncensored


Load data and constants

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
df = pd.read_csv(RAW / "CASComp Main DataSet.csv")

Create frequency and claims data frames

In [ ]:
policy_cols = ['student_id','name','class','study','gpa','greek','off_campus',
               'distance_to_campus','gender','sprinklered','risk_tier','training_holdout']

freq = (df.assign(is_claim=(df.claim_id > 0).astype(int))
          .groupby(policy_cols + ['coverage'], as_index=False, observed=True)
          .agg(claim_count=('is_claim','sum'),
               total_amount=('amount','sum')))

claims = df[df.claim_id > 0].copy()


print(claims.shape)
display(freq.head())
display(claims.head())



-

write to csv for future use

In [ ]:
freq.to_csv(PROC / "frequency_data.csv", index=False)
claims.to_csv(PROC / "claims.csv", index=False)

print(sorted(p.name for p in PROC.iterdir()))